# Chapter 17: Building an ODE Solver Visualizer

Companion notebook for *Python by Curiosity*. Every code listing from the chapter is
here, in the order it appears in the book, under the same title. Run the cells from
top to bottom: some listings use variables or functions defined in earlier cells.

The same listings are also available as separate scripts in `codes/listings/ch17_ode_visualizer/`,
and the exercise solutions are in `solutions/`.

**These listings open desktop windows (Tkinter).** Run the notebook on your own
computer (Jupyter or VS Code), not in an online service such as Google Colab. Each
cell waits until you close its window; close it to move on to the next cell.

In [ ]:
# Run the listings from the repository root so that paths such as
# data/grades.csv and figures/trophy.png work.
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("Working folder:", os.getcwd())

### Code 17.1: Scientific ODE solver visualizer

In [ ]:
import tkinter as tk
from tkinter import messagebox
import numpy as np
import matplotlib
matplotlib.use("TkAgg")
from matplotlib.figure import Figure
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg

def rk4(f, y0, t_start, t_end, h):
    t = np.arange(t_start, t_end + h, h)
    y = np.zeros(len(t))
    y[0] = y0
    for i in range(len(t) - 1):
        k1 = f(t[i], y[i])
        k2 = f(t[i] + h / 2, y[i] + h / 2 * k1)
        k3 = f(t[i] + h / 2, y[i] + h / 2 * k2)
        k4 = f(t[i] + h, y[i] + h * k3)
        y[i + 1] = y[i] + h / 6 * (k1 + 2 * k2 + 2 * k3 + k4)
    return t, y

class OdeVisualizerApp:
    def __init__(self, root):
        self.root = root
        self.root.title("ODE Solver Visualizer")
        self.root.geometry("600x500")
        self.model = lambda t, y: -0.3 * y

        # Left panel: parameters
        self.left_panel = tk.Frame(root, width=200, bg="lightgrey")
        self.left_panel.pack(side=tk.LEFT, fill=tk.Y, padx=5, pady=5)
        self.create_inputs()

        # Right panel: matplotlib canvas
        self.right_panel = tk.Frame(root)
        self.right_panel.pack(side=tk.RIGHT, fill=tk.BOTH, expand=True, padx=5, pady=5)
        self.setup_plot()

    def create_inputs(self):
        tk.Label(self.left_panel, text="y0 (Initial):", bg="lightgrey").pack(pady=(10, 2))
        self.ent_y0 = tk.Entry(self.left_panel, width=15)
        self.ent_y0.insert(0, "10.0")
        self.ent_y0.pack()

        tk.Label(self.left_panel, text="Step Size (h):", bg="lightgrey").pack(pady=2)
        self.ent_h = tk.Entry(self.left_panel, width=15)
        self.ent_h.insert(0, "0.5")
        self.ent_h.pack()

        tk.Label(self.left_panel, text="t Start:", bg="lightgrey").pack(pady=2)
        self.ent_tstart = tk.Entry(self.left_panel, width=15)
        self.ent_tstart.insert(0, "0.0")
        self.ent_tstart.pack()

        tk.Label(self.left_panel, text="t End:", bg="lightgrey").pack(pady=2)
        self.ent_tend = tk.Entry(self.left_panel, width=15)
        self.ent_tend.insert(0, "15.0")
        self.ent_tend.pack()

        self.btn_solve = tk.Button(
            self.left_panel, text="Solve & Plot", command=self.solve_and_plot
        )
        self.btn_solve.pack(pady=20)

    def setup_plot(self):
        self.fig = Figure(figsize=(5, 4), dpi=100)
        self.ax = self.fig.add_subplot(111)
        self.ax.set_title("System Trajectory y(t)")
        self.ax.set_xlabel("t")
        self.ax.set_ylabel("y")
        self.ax.grid(True)

        self.canvas = FigureCanvasTkAgg(self.fig, master=self.right_panel)
        self.canvas.get_tk_widget().pack(fill=tk.BOTH, expand=True)

    def solve_and_plot(self):
        try:
            y0 = float(self.ent_y0.get())
            h = float(self.ent_h.get())
            t_start = float(self.ent_tstart.get())
            t_end = float(self.ent_tend.get())

            if h <= 0:
                raise ValueError("Step size h must be positive.")
            if t_end <= t_start:
                raise ValueError("t End must be greater than t Start.")

            t, y = rk4(self.model, y0, t_start, t_end, h)

            self.ax.clear()
            self.ax.plot(t, y, marker="o", linestyle="-", label=f"RK4 (h={h})")
            self.ax.set_title("System Trajectory y(t)")
            self.ax.set_xlabel("t")
            self.ax.set_ylabel("y")
            self.ax.grid(True)
            self.ax.legend()
            self.canvas.draw()
        except ValueError as err:
            messagebox.showerror("Input Error", f"Invalid parameters:\n{err}")

if __name__ == "__main__":
    root = tk.Tk()
    app = OdeVisualizerApp(root)
    root.mainloop()

### Worked Example 17.1: Interactive Damped Harmonic Oscillator Visualizer

In [ ]:
import tkinter as tk
import numpy as np
import matplotlib
matplotlib.use("TkAgg")
from matplotlib.figure import Figure
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg

def solve_oscillator(zeta, omega_n=2.0, t_max=10.0, dt=0.02):
    t = np.arange(0, t_max, dt)
    # State vector: Y = [position x, velocity v]
    # dY/dt = [v, -2*zeta*omega_n*v - omega_n^2*x]
    x = np.zeros(len(t))
    v = np.zeros(len(t))
    x[0], v[0] = 1.0, 0.0  # Initial displacement of 1.0 unit
    
    for i in range(len(t) - 1):
        # Semi-implicit Euler: update v first, then use the NEW v for x.
        a = -2.0 * zeta * omega_n * v[i] - (omega_n ** 2) * x[i]
        v[i + 1] = v[i] + a * dt
        x[i + 1] = x[i] + v[i + 1] * dt
    return t, x

class OscillatorApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Damped Oscillator Visualizer")
        self.root.geometry("640x480")
        
        # Control Panel
        ctrl = tk.Frame(root, bg="#2c3e50", padx=10, pady=10)
        ctrl.pack(side="top", fill="x")
        
        tk.Label(ctrl, text="Damping Ratio (zeta):", font=("Arial", 10, "bold"),
                 fg="white", bg="#2c3e50").pack(side="left", padx=5)
                 
        self.slider_zeta = tk.Scale(ctrl, from_=0.0, to=2.0, resolution=0.05,
                                    orient="horizontal", length=250, command=self.update_plot)
        self.slider_zeta.set(0.15)
        self.slider_zeta.pack(side="left", padx=10)
        
        self.lbl_regime = tk.Label(ctrl, text="Regime: Underdamped", font=("Arial", 10, "bold"),
                                   fg="#f39c12", bg="#2c3e50")
        self.lbl_regime.pack(side="left", padx=10)
        
        # Matplotlib Canvas
        self.fig = Figure(figsize=(6, 4), dpi=100)
        self.ax = self.fig.add_subplot(111)
        self.canvas = FigureCanvasTkAgg(self.fig, master=root)
        self.canvas.get_tk_widget().pack(fill="both", expand=True)
        
        self.update_plot(self.slider_zeta.get())

    def update_plot(self, val):
        zeta = float(val)
        if zeta < 1.0:
            regime = "Underdamped (Oscillatory)"
            color = "#e74c3c"
        elif np.isclose(zeta, 1.0, atol=0.05):
            regime = "Critically Damped"
            color = "#27ae60"
        else:
            regime = "Overdamped (No Oscillation)"
            color = "#2980b9"
        self.lbl_regime.config(text=f"Regime: {regime}", fg=color)
        
        t, x = solve_oscillator(zeta)
        self.ax.clear()
        self.ax.plot(t, x, color=color, lw=2, label=f"zeta = {zeta:.2f}")
        self.ax.axhline(0, color="gray", linestyle="--", alpha=0.6)
        self.ax.set_title("Oscillator Displacement x(t)", fontsize=11, fontweight="bold")
        self.ax.set_xlabel("Time (s)")
        self.ax.set_ylabel("Displacement x")
        self.ax.set_ylim(-1.2, 1.2)
        self.ax.grid(True, alpha=0.3)
        self.ax.legend(loc="upper right")
        self.canvas.draw()

if __name__ == "__main__":
    root = tk.Tk()
    app = OscillatorApp(root)
    root.mainloop()

### Worked Example 17.2: Logistic Population Dynamics with Exact Comparison

In [ ]:
import tkinter as tk
import numpy as np
import matplotlib
matplotlib.use("TkAgg")
from matplotlib.figure import Figure
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg

class LogisticApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Logistic Population Growth")
        self.root.geometry("620x460")
        
        # Header controls
        pnl = tk.Frame(root, padx=10, pady=10)
        pnl.pack(side="top", fill="x")
        
        tk.Label(pnl, text="Growth Rate r:").pack(side="left")
        self.ent_r = tk.Entry(pnl, width=6)
        self.ent_r.insert(0, "0.5")
        self.ent_r.pack(side="left", padx=5)
        
        tk.Label(pnl, text="Carrying Capacity K:").pack(side="left", padx=(10, 0))
        self.ent_k = tk.Entry(pnl, width=6)
        self.ent_k.insert(0, "100")
        self.ent_k.pack(side="left", padx=5)
        
        btn = tk.Button(pnl, text="Simulate", font=("Arial", 9, "bold"),
                        bg="#27ae60", fg="white", command=self.simulate)
        btn.pack(side="left", padx=15)
        
        # Plot canvas
        self.fig = Figure(figsize=(6, 3.8), dpi=100)
        self.ax = self.fig.add_subplot(111)
        self.canvas = FigureCanvasTkAgg(self.fig, master=root)
        self.canvas.get_tk_widget().pack(fill="both", expand=True)
        self.simulate()

    def simulate(self):
        r = float(self.ent_r.get())
        K = float(self.ent_k.get())
        P0 = 5.0
        t = np.linspace(0, 15, 200)
        
        # Exact analytical solution
        P_exact = (K * P0 * np.exp(r * t)) / (K + P0 * (np.exp(r * t) - 1))
        
        # Numerical Euler solution with coarse step
        dt = 0.5
        t_num = np.arange(0, 15 + dt, dt)
        P_num = np.zeros(len(t_num))
        P_num[0] = P0
        for i in range(len(t_num) - 1):
            dP = r * P_num[i] * (1 - P_num[i] / K)
            P_num[i + 1] = P_num[i] + dP * dt
            
        self.ax.clear()
        self.ax.plot(t, P_exact, "b-", lw=2, label="Exact Analytical")
        self.ax.plot(t_num, P_num, "ro--", label="Numerical Euler (dt=0.5)")
        self.ax.axhline(K, color="green", linestyle=":", label=f"Carrying Capacity K={K}")
        self.ax.set_title("Logistic Growth: Numerical vs. Analytical", fontweight="bold")
        self.ax.set_xlabel("Time")
        self.ax.set_ylabel("Population P(t)")
        self.ax.legend()
        self.ax.grid(True, alpha=0.3)
        self.canvas.draw()

if __name__ == "__main__":
    root = tk.Tk()
    app = LogisticApp(root)
    root.mainloop()